In [ ]:
from univmon import UnivMon
import math
import time
import pandas as pd
import scipy as sp
import numpy as np
from tqdm import tqdm

In [ ]:
path = 'path/to/appraise.csv'
df = pd.read_csv(path)
df

,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS
0,3.257485e+09,20828.0,3.260381e+09,52.0,18.0,179430.0,2955.0,272656.0,2576.0
1,3.253771e+09,19149.0,3.249148e+09,466.0,17.0,26506.0,3318.0,2749168.0,2930.0
2,2.895684e+09,52561.0,2.916716e+09,276.0,7.0,1205313.0,3052.0,792420.0,1013.0
3,3.232337e+09,62676.0,3.241048e+09,174.0,6.0,2995190.0,1983.0,2458753.0,2300.0
4,2.899580e+09,62599.0,2.909405e+09,439.0,5.0,149998.0,389.0,280651.0,2348.0
...,...,...,...,...,...,...,...,...,...
75987971,3.240715e+09,46722.0,3.236086e+09,50072.0,7.0,594195.0,1498.0,267200.0,2088.0
75987972,3.253769e+09,49278.0,3.243913e+09,1315.0,7.0,984634.0,2500.0,1443692.0,3566.0
75987973,3.236313e+09,55482.0,3.224522e+09,407.0,7.0,788061.0,605.0,1188864.0,3345.0
75987974,3.232314e+09,41090.0,3.224105e+09,598.0,6.0,2522048.0,3074.0,2310995.0,1320.0


In [3]:
um = UnivMon(mem_in_bytes=1_000_000, level=12, rows=5, k=1000) # 1MB

# UnivMon Evaluation

In [3]:
import statistics
import os
import json
from copy import deepcopy
import concurrent
from tqdm.contrib.concurrent import process_map

In [4]:
SAVE_FOLDER = 'eval/univmon/'
NAME = 'nfiot_dp'
os.makedirs(SAVE_FOLDER, exist_ok=True)

In [5]:
univmons = {}

def compute_sketch(idx, data):
    um = UnivMon(mem_in_bytes=1_000_000, level=12, rows=5, k=1000) # 1MB
    for _, item in data.items():
        um.insert(item)
    return um

job_args = [(i, col) for i, col in enumerate(df.columns)]
with tqdm(total=len(job_args)) as pbar:
    # let's give it some more threads:
    with concurrent.futures.ProcessPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(compute_sketch, i, df[col]): col for i, col in job_args}
        results = {}
        for future in concurrent.futures.as_completed(futures):
            col = futures[future]
            results[col] = future.result()
            pbar.update(1)

print(f'{len(df.columns)} Features...')
univmons = results
univmons

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [19:33<00:00, 130.35s/it]

9 Features...


{'PROTOCOL': <univmon.UnivMon at 0x7f41b6c47560>,
 'L4_DST_PORT': <univmon.UnivMon at 0x7f41b6c47a40>,
 'L4_SRC_PORT': <univmon.UnivMon at 0x7f41b6c47d10>,
 'IPV4_SRC_ADDR': <univmon.UnivMon at 0x7f41b6c47a70>,
 'IPV4_DST_ADDR': <univmon.UnivMon at 0x7f41b6c46780>,
 'OUT_BYTES': <univmon.UnivMon at 0x7f41b6c454c0>,
 'OUT_PKTS': <univmon.UnivMon at 0x7f41b6c47470>,
 'IN_PKTS': <univmon.UnivMon at 0x7f41b6c47fe0>,
 'IN_BYTES': <univmon.UnivMon at 0x7f41b6c47530>}

## Entropy

In [7]:
entropies = {}

def eval_entropy(gt_data, sketch):
    val_freqs = gt_data.value_counts()
    real_entropy = sp.stats.entropy(val_freqs)
    sketch_entropy = sketch.get_entropy(val_freqs.index)
    entropy = abs((real_entropy - sketch_entropy) / real_entropy)
    return entropy
    

job_args = [(df[col], univmons[col], col) for col in df.columns]
with tqdm(total=len(job_args)) as pbar:
    # let's give it some more threads:
    with concurrent.futures.ProcessPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(eval_entropy, gt, sketch): col for gt, sketch, col in job_args}
        results = {}
        for future in concurrent.futures.as_completed(futures):
            col = futures[future]
            results[col] = future.result()
            pbar.update(1)
            
entropies = results
entropies, statistics.mean(entropies.values()), statistics.stdev(entropies.values())

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [1:16:17<00:00, 508.62s/it]


({'PROTOCOL': 0.44269504088897516,
  'L4_SRC_PORT': 0.43704544224659503,
  'L4_DST_PORT': 0.48493310840189674,
  'IN_PKTS': 0.35037245810470213,
  'OUT_PKTS': 0.4207953768636835,
  'OUT_BYTES': 0.42118421985579296,
  'IN_BYTES': 0.4406962112316769,
  'IPV4_DST_ADDR': 0.43179009457548606,
  'IPV4_SRC_ADDR': 0.4292578144551178},
 0.42875219629154737,
 0.03506349294328225)

In [8]:
path = os.path.join(SAVE_FOLDER, f'{NAME}_entropy.json')
out = deepcopy(entropies)

with open(path, 'w') as f:
    out.update({
        'mean': statistics.mean(entropies.values()),
        'std': statistics.stdev(entropies.values())
    })
    json.dump(out, f, indent=2)